In [ ]:
import itertools
from amplpy import AMPL

# ----------------------------
# Generate valid bound vectors
# ----------------------------
def generate_valid_vectors(N):
    """Generate all 0/1 vectors with the constraint: if vec[i] == 1 then vec[N+i] must be 1"""
    for vec in itertools.product([0, 1], repeat=2 ** (2* N)):
        if all(not (vec[i] == 1 and vec[N + i] == 0) for i in range(N)):
            yield tuple(vec)


# ----------------------------
# Solve relaxation for a given bound vector (uses AMPL commands to set bounds)
# ----------------------------
def solve_relaxation(mod_file, int_var_names, bound_vec, solver=None):
    ampl = AMPL()
    ampl.read(mod_file)
    if solver:
        ampl.setOption("solver", solver)

    n = len(int_var_names)
    for i, name in enumerate(int_var_names):
        lb = float(bound_vec[i])
        ub = float(bound_vec[n + i])
        var = ampl.getVariable(name)
        var.lb()= lb
        var.ub()= ub

    # Solve LP relaxation
    ampl.option["relax_integrality"] = 1
    ampl.solve()
    obj_val = ampl.getObjective(0).value()

    # collect values of integer vars (for branching decisions)
    sol = [var.value() for var in ampl.getVariables() if var.name() in int_var_names]

    ampl.close()
    return obj_val, sol


# ----------------------------
# Recursive + memoized branching
# ----------------------------
def branch_on_integers(mod_file, int_var_names, state, value_store, solver=None):
    """
    Compute V(state) = max_over_variables max_over_left/right [DeltaObj + V(s')]
    with memoization in value_store (dict from state->value).
    """
    # memoization lookup
    if state in value_store:
        return value_store[state]

    base_obj, _ = solve_relaxation(mod_file, int_var_names, state, solver=solver)
    best_value = float("-inf")
    best_var = None
    n = len(int_var_names)

    # Try branching on each variable (fix it to its lb or ub)
    for idx, name in enumerate(int_var_names):
        # left: fix to lower bound (set ub = lb)
        left_state = list(state)
        left_state[n + idx] = left_state[idx]  # set upper = lower (fix to lower)
        left_state = tuple(left_state)
        left_obj, left_new = solve_relaxation(mod_file, int_var_names, left_state, solver=solver)
        # recursively evaluate child (memoized) unless child equals current state
        left_future = 0.0 if left_new == state else branch_on_integers(mod_file, int_var_names, left_new, value_store, solver=solver)
        left_value = (left_obj - base_obj) + left_future

        # right: fix to upper bound (set lb = ub)
        right_state = list(state)
        right_state[idx] = right_state[n + idx]  # set lower = upper (fix to upper)
        right_state = tuple(right_state)
        right_obj, right_new = solve_relaxation(mod_file, int_var_names, right_state, solver=solver)
        right_future = 0.0 if right_new == state else branch_on_integers(mod_file, int_var_names, right_new, value_store, solver=solver)
        right_value = (right_obj - base_obj) + right_future

        # take max over left/right for this branching variable
        branch_value = max(left_value, right_value)

        if branch_value > best_value:
            best_value = branch_value
            best_var = name

    # store in memo table
    value_store[state] = best_value

    # print requested info
    print(f"State: {state}")
    print(f"  Best branching var: {best_var}")
    print(f"  V(s) = {best_value:.6f}\n")

    return best_value


# ----------------------------
# Main: prepare variable names once, then run
# ----------------------------
mod_file = "/home/kkishan/Downloads/alan.mod"   # <- change if needed
ampl = AMPL()
ampl.read(mod_file)

# ampl.getVariables() returns iterable of (name, entity) pairs in some versions -> take the names
int_var_names = [name for name, _ in ampl.getVariables()]
n = len(int_var_names)
ampl.close()

# generate initial bound vectors (you already had this)
bound_vectors = list(generate_valid_vectors(n))

# value store (memo)
value_store = {}

# run
for vec in bound_vectors:
    branch_on_integers(mod_file, int_var_names, tuple(vec), value_store, solver=None)

# optional: view the value store
print("Stored V(s) count:", len(value_store))


SyntaxError: cannot assign to function call here. Maybe you meant '==' instead of '='? (3167514750.py, line 28)